##Contexto

Neste curso, eu aplico técnica de HAG (Retrieval Argument Generation) para ampliar o contexto dos LLMs e gerar respostas mais inteligentes e precisas.


##### Conceitos de LLM

Existem milhões de modelos diferentes, mas, de forma geral, um LLM (Large Language Model) é um grande modelo de linguagem que trabalha com palavras transformadas em tokens. Dentro da sua arquitetura, ele busca padrões e contextos relacionados a esses tokens para gerar respostas coerentes e contextualizadas.

Seu funcionamento é baseado na previsão da próxima palavra, considerando todo o contexto da conversa ou do texto fornecido.

A principal arquitetura utilizada pelos LLMs é a Transformer, responsável por permitir que o modelo compreenda relações entre palavras e contextos de maneira muito mais eficiente.


##### RAG (Retrieval-Augmented Generation)
É uma técnica que permite que a IA busque informações externas antes de responder uma pergunta.
###https://cursos.alura.com.br/course/langchain-chatbots-rag/task/233596

##Agente Explicador de Regras do Futebol com RAG (LangChain)
######Objetivo: 
Construir um agente conversacional simples utilizando Retrieval-Augmented Generation (RAG) para responder perguntas sobre regras do futebol, com base em documentos oficiais.

#### Bibiotecas 

In [0]:
%pip install --upgrade chromadb langchain-openai

In [0]:
%pip install langchain-community langchain-openai chromadb pypdf
%pip uninstall -y langchain langchain-community langchain-openai pydantic typing_extensions

In [0]:
%pip install \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-openai==0.1.23 \
pydantic==2.8.2 \
typing_extensions==4.12.2 \
chromadb \
pypdf

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from pypdf import PdfReader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

##### Objetivo

LLMs possuem conhecimento estático e podem alucinar. O objetivo aqui é garantir respostas confiáveis, conectando o modelo a documentos oficiais sobre regras do futebol

In [0]:
##leitura de arquivos
CAMINHO_PDF = "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/regras_futebol.pdf"
loader = PyPDFLoader(CAMINHO_PDF)
documents = loader.load()
len(documents)

In [0]:
documents 

####Preparação dos Documentos | Chunks

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

len(chunks)

###Aqui os dados vira vetores para ser embeddings

In [0]:
chunks ## mostrar 3 pedaços

####Embeddings e Banco Vetorial

In [0]:
pip install langchain-community faiss-cpu

In [0]:
# Inicializa embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key="xx"
)

In [0]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
local_path = "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/faiss_index"

vectorstore.save_local(local_path)

####Recuperação de Contexto (Retriever)

In [0]:
# Cria o retriever || retriver busca os trechos masi relevantes para cada pergunta do usuario
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [0]:
###O contexto recuperado será injetado no prompt enviado ao modelo de linguagem

llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key="xxx"
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

#### Testes e Validação

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "xx"

pergunta = "Um jogador pode usar a mão para marcar um gol?"
resposta = qa_chain.invoke({"query": pergunta})

print("Pergunta:")
print(pergunta)
print("\nResposta do Agente:")
print(resposta.get("result", resposta.get("answer")))
print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta.get("source_documents", []), start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("Conteúdo:")
    print(doc.page_content)
    print("\n")